# Week 10 Day 2 lab | Insert and find MongoDB documents

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_10/materials/notebooks/10_day_2_insert_and_find.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_10/materials/notebooks/10_day_2_insert_and_find.ipynb) · [Student handout](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_10/lab_02_document_model.md) · [Chapter 10 reading response](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_10/reading_response.md)

You work a ticket queue. Which tickets should staff see when they ask for open requests at least 30 minutes old? Create a ticket, read it back, and test your answer against a collection. Keep three short notes for Part A as you work. Allow **55 minutes**, then switch to the assigned reading.

In Colab, connect to a Python runtime. In local Jupyter, open this notebook with a Python 3.10 or newer kernel. Run one cell at a time with **Shift + Enter** so you can predict each result before it appears. All ticket records in this exercise are fictional.

## 1. Open your practice database | 5 minutes

Run the setup cell. It installs the Python driver if needed and starts a temporary MongoDB Community Server in this notebook's runtime. The first run may download about 100 MB from MongoDB. Look for **MongoDB Community Server** in the result. If the download or server start fails, the notebook uses **in-memory simulation (mongomock)** and prints that route. If you have a slow connection, set `use_simulation` to `True` before running the cell. Copy the printed route into Part A. You can use either route without an Atlas account.

In [ ]:
# @title Start the practice database
use_simulation = False  # @param {type:"boolean"}
import atexit
import hashlib
from http.client import HTTPException
import importlib.util
import platform
import shutil
import socket
import subprocess
import sys
import tarfile
import tempfile
import time
from contextlib import ExitStack
from pathlib import Path
from urllib.request import urlopen

requirements = {"pymongo": "pymongo>=4.13,<5", "mongomock": "mongomock>=4.3,<5"}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import mongomock
from pymongo import MongoClient
from pymongo.errors import DuplicateKeyError, PyMongoError


def mongo_archive():
    arch = {"arm64": "aarch64", "AMD64": "x86_64"}.get(platform.machine(), platform.machine())
    if platform.system() == "Darwin":
        suffix = "arm64" if arch == "aarch64" else arch
        return f"osx/mongodb-macos-{suffix}-8.0.14.tgz"
    if platform.system() == "Linux":
        release = platform.freedesktop_os_release()
        releases = {("ubuntu", "22"): "ubuntu2204", ("ubuntu", "24"): "ubuntu2404", ("debian", "12"): "debian12"}
        variant = releases.get((release.get("ID"), release.get("VERSION_ID", "").split(".")[0]))
        if variant:
            return f"linux/mongodb-linux-{arch}-{variant}-8.0.14.tgz"
    raise OSError("No automatic MongoDB download for this platform")


def launch_server():
    binary = shutil.which("mongod")
    if not binary:
        url = "https://fastdl.mongodb.org/" + mongo_archive()
        archive = scratch / "server.tgz"
        with urlopen(url + ".sha256", timeout=15) as source:
            expected = source.read().decode().split()[0]
        digest = hashlib.sha256()
        deadline = time.monotonic() + 90
        with urlopen(url, timeout=20) as source, archive.open("wb") as target:
            while chunk := source.read(1 << 20):
                if time.monotonic() > deadline:
                    raise TimeoutError("MongoDB download timed out")
                digest.update(chunk)
                target.write(chunk)
        if digest.hexdigest() != expected:
            raise OSError("The server download did not match MongoDB’s checksum")
        binary = scratch / "mongod"
        with tarfile.open(archive) as tar:
            member = next(m for m in tar if m.isfile() and m.name.endswith("/bin/mongod"))
            with tar.extractfile(member) as source, binary.open("wb") as target:
                shutil.copyfileobj(source, target)
        binary.chmod(0o700)
        archive.unlink()
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        port = sock.getsockname()[1]
    data = scratch / "data"
    data.mkdir()
    diagnostics = scratch / "startup.log"
    with diagnostics.open("w") as errors:
        proc = subprocess.Popen([str(binary), "--dbpath", str(data), "--bind_ip", "127.0.0.1", "--port", str(port),
                                 "--logpath", str(scratch / "mongod.log")], stdout=subprocess.DEVNULL, stderr=errors)
    def stop():
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
    session.callback(stop)
    client = MongoClient(f"mongodb://127.0.0.1:{port}", serverSelectionTimeoutMS=8000)
    try:
        return client, "MongoDB Community Server " + client.server_info()["version"]
    except Exception:
        client.close()
        stop()
        detail = diagnostics.read_text(errors="replace").strip()
        if detail:
            raise OSError(detail.splitlines()[-1]) from None
        raise


# Restart this cell to work in a fresh, isolated database.
if "session" in globals():
    session.close()
session = ExitStack()
atexit.register(session.close)
scratch = Path(session.enter_context(tempfile.TemporaryDirectory(prefix="week10_day2_")))
if use_simulation:
    db_client, route = mongomock.MongoClient(), "in-memory simulation (mongomock)"
else:
    try:
        db_client, route = launch_server()
    except (OSError, HTTPException, ValueError, PyMongoError, tarfile.TarError, StopIteration) as exc:
        print("Could not start the server:", str(exc).splitlines()[0][:180])
        db_client, route = mongomock.MongoClient(), "in-memory simulation (mongomock)"
session.callback(db_client.close)
tickets = db_client.week10_day2.practice
print("Practice route:", route)
print("Database and collection:", tickets.full_name)

## 2. Read the queue and insert a ticket | 15 minutes

Run the seed cell once. Each row has a ticket number, a status, and an age in minutes. The ages are specific to this exercise. Rerunning this cell resets only this notebook’s practice collection. Look at the seed rows. Which document is exactly 30 minutes old? Is it open?

In [ ]:
seed = [
    {"ticket_id": 1001, "status": "open",        "age_minutes": 12},
    {"ticket_id": 1002, "status": "in_progress", "age_minutes": 30},
    {"ticket_id": 1003, "status": "resolved",    "age_minutes": 45},
]
tickets.delete_many({})
tickets.create_index("ticket_id", unique=True)
tickets.insert_many([row.copy() for row in seed])
for row in tickets.find({}, {"_id": 0}).sort("ticket_id", 1):
    print(row)

A resident opens ticket **1090**. It has been waiting **30 minutes**. Change the `subject` value to a short fictional description of your choice before you run the cell. Predict which field MongoDB will add for you. Keep the returned ID so you can retrieve this exact write.

In [ ]:
new_ticket = {"ticket_id": 1090, "status": "open", "age_minutes": 30,
              "subject": "Replace this with your own fictional issue"}
try:
    receipt = tickets.insert_one(new_ticket.copy())
    stored_id = receipt.inserted_id
    print("Inserted _id:", stored_id)
except DuplicateKeyError:
    stored_id = tickets.find_one({"ticket_id": 1090})["_id"]
    print("Ticket 1090 was already inserted. Reading the existing record:", stored_id)

Read the ticket back using the returned ID. Check whether the `subject` matches what you wrote. Now change the second lookup to a ticket number of your choice. Which lookup returns a document? What does `None` tell you? Record the read-back filter and one field/value for Part A. To change the subject after an insert, reset the seed and rerun the insert.

In [ ]:
record = tickets.find_one({"_id": stored_id})
print("Read back by _id:", record)
other_number = 9999  # Try a ticket number that exists, then a number that does not.
print("Second lookup:", tickets.find_one({"ticket_id": other_number}, {"_id": 0}))

## 3. Change which tickets appear | 15 minutes

A dispatcher wants a list of open tickets. Predict the matching ticket IDs from memory, then use the function below to display them. The projection returns the ticket number, and the sort makes it easier to compare runs. Change `status` to another value you've seen and run the cell again. Keep both sets of results.

In [ ]:
def matching_ids(condition):
    rows = tickets.find(condition, {"_id": 0, "ticket_id": 1}).sort("ticket_id", 1)
    return [row["ticket_id"] for row in rows]

status = "open"
status_filter = {"status": status}
print(status_filter, "→", matching_ids(status_filter))

The dispatcher narrows the request to **open and at least 30 minutes old**. Write down which IDs you expect. Run the cell and inspect the documents if a result surprises you. Which condition excludes the younger open ticket? Which condition excludes the 30-minute ticket that was already in the collection?

In [ ]:
threshold = 30
inclusive = {"status": "open", "age_minutes": {"$gte": threshold}}
print(inclusive, "→", matching_ids(inclusive))
for row in tickets.find({}, {"_id": 0, "ticket_id": 1, "status": 1, "age_minutes": 1}).sort("ticket_id", 1):
    print(row)

## 4. Test the boundary | 10 minutes

Keep the status and age the same. Replace `"$gte"` with `"$gt"` in the cell below and predict what will happen to the ticket exactly at 30. Run the comparison. Record both full filters and both results. An empty list is a result. Explain the difference in one sentence for Part A.

In [ ]:
operator = "$gte"  # Edit to "$gt" for the second query; keep cutoff at 30.
cutoff = 30
changed = {"status": "open", "age_minutes": {operator: cutoff}}
print("Original:", inclusive, "→", matching_ids(inclusive))
print("Changed:", changed, "→", matching_ids(changed))

A teammate says a strict boundary will always produce an empty result here. Change only `cutoff` so at least one open ticket qualifies. Test your choice in the same cell and save this third result separately. If two results look identical, check the operators, cutoff, and the types of the stored ages.

## 5. Limit the display and collect evidence | 10 minutes

Build a screen that displays the **two oldest tickets**, ordered by age. Predict which documents it will display. Run the cell, then change `page_size` to `1`. Find one ticket that dropped off the screen and look it up by ID in the next cell. What experiment tells you that changing a display limit preserves the stored records?

In [ ]:
page_size = 2
queue = tickets.find({}, {"_id": 0, "ticket_id": 1, "age_minutes": 1}).sort(
    [("age_minutes", -1), ("ticket_id", 1)]
).limit(page_size)
print("Displayed:", list(queue))
print("Stored document count:", tickets.count_documents({}))

In [ ]:
outside_page = 1001  # Edit to an ID omitted from your latest display.
print("Separate lookup:", tickets.find_one({"ticket_id": outside_page}, {"_id": 0}))

**Part A.** In your Brightspace entry, name the route printed by the setup cell and the last step you reached. Give the collection, the field/value you inserted, the filter you used to retrieve the write, and one returned value. Show your original and changed filters, their actual results, and why one ticket appeared or disappeared.

**Part B.** Switch to the [Chapter 10 reading response](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_10/reading_response.md). In **150–220 words**, use the assigned section and a result you observed here to explain your design for ticket 1003. Submit Parts A and B together. Copy your observations before closing the notebook.